In [1]:
import pandas as pd
import numpy as np
import time
import json
import os
import joblib

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import OneHotEncoder

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

print("XGBoost version:", xgb.__version__)
print("LightGBM version:", lgb.__version__)
print("CatBoost version:", __import__('catboost').__version__)

XGBoost version: 3.4.1
LightGBM version: 4.7.0
CatBoost version: 1.2.10


In [2]:
data = load_breast_cancer(as_frame=True)

X = data.data
y = data.target

print("Размер датасета:", X.shape)
print("Количество признаков:", X.shape[1])
print("Классы:", data.target_names)

print("\nРаспределение классов:")
print(y.value_counts().rename(index={
    0: "malignant",
    1: "benign"
}))

display(X.head())

Размер датасета: (569, 30)
Количество признаков: 30
Классы: ['malignant' 'benign']

Распределение классов:
target
benign       357
malignant    212
Name: count, dtype: int64


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=67,
    stratify=y
)
print("Обучающая выборка:", X_train.shape)
print("Тестовая выборка:", X_test.shape)

X_train_val, X_val, y_train_val, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=67,
    stratify=y_train
)
print("Обучающая для бустинга:", X_train_val.shape)
print("Валидационная для ранней остановки:", X_val.shape)

Обучающая выборка: (455, 30)
Тестовая выборка: (114, 30)
Обучающая для бустинга: (364, 30)
Валидационная для ранней остановки: (91, 30)


In [4]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=67,
    n_jobs=-1
)
start = time.time()
rf.fit(X_train, y_train)
rf_train_time = time.time() - start

rf_pred_train = rf.predict(X_train)
rf_pred_test = rf.predict(X_test)

rf_results = {
    "model": "Random Forest",
    "accuracy_train": accuracy_score(y_train, rf_pred_train),
    "accuracy_test": accuracy_score(y_test, rf_pred_test),
    "f1_macro_train": f1_score(y_train, rf_pred_train, average="macro"),
    "f1_macro_test": f1_score(y_test, rf_pred_test, average="macro"),
    "train_time": rf_train_time
}

print("Random Forest обучен")
print(f"Время обучения: {rf_train_time:.3f} сек")
print(f"Accuracy test: {rf_results['accuracy_test']:.4f}")
print(f"F1 macro test: {rf_results['f1_macro_test']:.4f}")

Random Forest обучен
Время обучения: 0.084 сек
Accuracy test: 0.9386
F1 macro test: 0.9337


In [5]:
lr = LogisticRegression(
    max_iter=1000,
    random_state=67,
    n_jobs=-1
)
start = time.time()
lr.fit(X_train, y_train)
lr_train_time = time.time() - start

lr_pred_train = lr.predict(X_train)
lr_pred_test = lr.predict(X_test)

lr_results = {
    "model": "Logistic Regression",
    "accuracy_train": accuracy_score(y_train, lr_pred_train),
    "accuracy_test": accuracy_score(y_test, lr_pred_test),
    "f1_macro_train": f1_score(y_train, lr_pred_train, average="macro"),
    "f1_macro_test": f1_score(y_test, lr_pred_test, average="macro"),
    "train_time": lr_train_time
}

print("Logistic Regression обучена")
print(f"Время обучения: {lr_train_time:.3f} сек")
print(f"Accuracy test: {lr_results['accuracy_test']:.4f}")
print(f"F1 macro test: {lr_results['f1_macro_test']:.4f}")

Logistic Regression обучена
Время обучения: 0.123 сек
Accuracy test: 0.9211
F1 macro test: 0.9147


C:\Users\user\Desktop\projects\MLProjectsCS32\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
C:\Users\user\Desktop\projects\MLProjectsCS32\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [6]:
xgb_model = xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=4,
    random_state=67,
    n_jobs=-1,
    eval_metric="logloss",
    early_stopping_rounds=50,
    verbosity=0
)
start = time.time()
xgb_model.fit(
    X_train_val, y_train_val,
    eval_set=[(X_val, y_val)],
    verbose=False
)
xgb_train_time = time.time() - start

print(f"XGBoost обучен")
print(f"Время обучения: {xgb_train_time:.3f} сек")
print(f"Лучшая итерация: {xgb_model.best_iteration}")
print(f"Best score: {xgb_model.best_score:.4f}")

XGBoost обучен
Время обучения: 0.150 сек
Лучшая итерация: 116
Best score: 0.1612


In [7]:
xgb_pred_train = xgb_model.predict(X_train)
xgb_pred_test = xgb_model.predict(X_test)

xgb_results = {
    "model": "XGBoost",
    "accuracy_train": accuracy_score(y_train, xgb_pred_train),
    "accuracy_test": accuracy_score(y_test, xgb_pred_test),
    "f1_macro_train": f1_score(y_train, xgb_pred_train, average="macro"),
    "f1_macro_test": f1_score(y_test, xgb_pred_test, average="macro"),
    "train_time": xgb_train_time,
    "n_estimators_used": xgb_model.best_iteration + 1
}

print(f"Accuracy train: {xgb_results['accuracy_train']:.4f}")
print(f"Accuracy test: {xgb_results['accuracy_test']:.4f}")
print(f"F1 macro train: {xgb_results['f1_macro_train']:.4f}")
print(f"F1 macro test: {xgb_results['f1_macro_test']:.4f}")
print(f"Число деревьев: {xgb_results['n_estimators_used']}")

Accuracy train: 0.9912
Accuracy test: 0.9474
F1 macro train: 0.9906
F1 macro test: 0.9429
Число деревьев: 117


In [8]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=4,
    random_state=67,
    n_jobs=-1,
    verbosity=-1,
    force_col_wise=True
)
start = time.time()
lgb_model.fit(
    X_train_val, y_train_val,
    eval_set=[(X_val, y_val)],
    eval_metric="logloss",
    callbacks=[lgb.early_stopping(50, verbose=False)]
)
lgb_train_time = time.time() - start

print(f"LightGBM обучен")
print(f"Время обучения: {lgb_train_time:.3f} сек")
print(f"Лучшая итерация: {lgb_model.best_iteration_}")
best_score_key = list(lgb_model.best_score_['valid_0'].keys())[0]
print(f"Best score: {lgb_model.best_score_['valid_0'][best_score_key]:.4f}")

LightGBM обучен
Время обучения: 0.034 сек
Лучшая итерация: 116
Best score: 0.1276


C:\Users\user\Desktop\projects\MLProjectsCS32\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


In [9]:
lgb_pred_train = lgb_model.predict(X_train)
lgb_pred_test = lgb_model.predict(X_test)

lgb_results = {
    "model": "LightGBM",
    "accuracy_train": accuracy_score(y_train, lgb_pred_train),
    "accuracy_test": accuracy_score(y_test, lgb_pred_test),
    "f1_macro_train": f1_score(y_train, lgb_pred_train, average="macro"),
    "f1_macro_test": f1_score(y_test, lgb_pred_test, average="macro"),
    "train_time": lgb_train_time,
    "n_estimators_used": lgb_model.best_iteration_ + 1
}

print(f"Accuracy train: {lgb_results['accuracy_train']:.4f}")
print(f"Accuracy test: {lgb_results['accuracy_test']:.4f}")
print(f"F1 macro train: {lgb_results['f1_macro_train']:.4f}")
print(f"F1 macro test: {lgb_results['f1_macro_test']:.4f}")
print(f"Число деревьев: {lgb_results['n_estimators_used']}")

Accuracy train: 0.9934
Accuracy test: 0.9386
F1 macro train: 0.9930
F1 macro test: 0.9337
Число деревьев: 117


In [10]:
cat_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=4,
    random_seed=67,
    verbose=False,
    early_stopping_rounds=50,
    thread_count=-1
)
start = time.time()
cat_model.fit(
    X_train_val, y_train_val,
    eval_set=(X_val, y_val),
    verbose=False
)
cat_train_time = time.time() - start

print(f"CatBoost обучен")
print(f"Время обучения: {cat_train_time:.3f} сек")
print(f"Лучшая итерация: {cat_model.best_iteration_}")
print(f"Best score: {cat_model.best_score_['validation']['Logloss']:.4f}")

CatBoost обучен
Время обучения: 0.343 сек
Лучшая итерация: 154
Best score: 0.1433


In [11]:
cat_pred_train = cat_model.predict(X_train)
cat_pred_test = cat_model.predict(X_test)

cat_results = {
    "model": "CatBoost",
    "accuracy_train": accuracy_score(y_train, cat_pred_train),
    "accuracy_test": accuracy_score(y_test, cat_pred_test),
    "f1_macro_train": f1_score(y_train, cat_pred_train, average="macro"),
    "f1_macro_test": f1_score(y_test, cat_pred_test, average="macro"),
    "train_time": cat_train_time,
    "n_estimators_used": cat_model.best_iteration_ + 1
}

print(f"Accuracy train: {cat_results['accuracy_train']:.4f}")
print(f"Accuracy test: {cat_results['accuracy_test']:.4f}")
print(f"F1 macro train: {cat_results['f1_macro_train']:.4f}")
print(f"F1 macro test: {cat_results['f1_macro_test']:.4f}")
print(f"Число деревьев: {cat_results['n_estimators_used']}")

Accuracy train: 0.9934
Accuracy test: 0.9561
F1 macro train: 0.9930
F1 macro test: 0.9521
Число деревьев: 155


In [12]:
inference_results = {}

for name, model in [("XGBoost", xgb_model), ("LightGBM", lgb_model), ("CatBoost", cat_model), ("Random Forest", rf), ("Logistic Regression", lr)]:
    n_runs = 10
    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        _ = model.predict(X_test)
        times.append(time.perf_counter() - start)
    inference_results[name] = {
        "mean_time_ms": np.mean(times) * 1000,
        "std_time_ms": np.std(times) * 1000
    }

for name, res in inference_results.items():
    print(f"{name}: {res['mean_time_ms']:.2f} ± {res['std_time_ms']:.2f} ms")

XGBoost: 1.41 ± 0.10 ms
LightGBM: 0.87 ± 0.09 ms
CatBoost: 0.95 ± 0.06 ms
Random Forest: 31.89 ± 1.58 ms
Logistic Regression: 0.90 ± 0.20 ms


In [13]:
model_sizes = {}

os.makedirs("../models", exist_ok=True)

xgb_model.save_model("../models/xgb_model.json")
model_sizes["XGBoost"] = os.path.getsize("../models/xgb_model.json") / 1024

lgb_model.booster_.save_model("../models/lgb_model.txt")
model_sizes["LightGBM"] = os.path.getsize("../models/lgb_model.txt") / 1024

cat_model.save_model("../models/cat_model.cbm")
model_sizes["CatBoost"] = os.path.getsize("../models/cat_model.cbm") / 1024

joblib.dump(rf, "../models/rf_model.pkl")
model_sizes["Random Forest"] = os.path.getsize("../models/rf_model.pkl") / 1024

joblib.dump(lr, "../models/lr_model.pkl")
model_sizes["Logistic Regression"] = os.path.getsize("../models/lr_model.pkl") / 1024

for name, size in model_sizes.items():
    print(f"{name}: {size:.2f} KB")

XGBoost: 158.32 KB
LightGBM: 132.27 KB
CatBoost: 67.70 KB
Random Forest: 301.10 KB
Logistic Regression: 1.84 KB


In [14]:
all_results = [rf_results, lr_results, xgb_results, lgb_results, cat_results]

comparison_df = pd.DataFrame(all_results)
comparison_df = comparison_df[[
    "model",
    "accuracy_train",
    "accuracy_test",
    "f1_macro_train",
    "f1_macro_test",
    "train_time",
    "n_estimators_used"
]]
comparison_df.columns = ["Модель", "Accuracy train", "Accuracy test", "F1 macro train", "F1 macro test", "Время обучения (сек)", "Число деревьев"]

display(comparison_df.round(4))

,Модель,Accuracy train,Accuracy test,F1 macro train,F1 macro test,Время обучения (сек),Число деревьев
0,Random Forest,1.0000,0.9386,1.0000,0.9337,0.0844,NaN
1,Logistic Regression,0.9626,0.9211,0.9599,0.9147,0.1229,NaN
2,XGBoost,0.9912,0.9474,0.9906,0.9429,0.1501,117.0
3,LightGBM,0.9934,0.9386,0.9930,0.9337,0.0337,117.0
4,CatBoost,0.9934,0.9561,0.9930,0.9521,0.3430,155.0


In [15]:
inference_df = pd.DataFrame([
    {"Модель": name, "Время инференса (мс)": res["mean_time_ms"], "Std (мс)": res["std_time_ms"]}
    for name, res in inference_results.items()
])

size_df = pd.DataFrame([
    {"Модель": name, "Размер модели (KB)": size}
    for name, size in model_sizes.items()
])

full_comparison = comparison_df.merge(inference_df, on="Модель").merge(size_df, on="Модель")

display(full_comparison.round(4))

,Модель,Accuracy train,Accuracy test,F1 macro train,F1 macro test,Время обучения (сек),Число деревьев,Время инференса (мс),Std (мс),Размер модели (KB)
0,Random Forest,1.0000,0.9386,1.0000,0.9337,0.0844,NaN,31.8918,1.5773,301.1025
1,Logistic Regression,0.9626,0.9211,0.9599,0.9147,0.1229,NaN,0.9037,0.2008,1.8428
2,XGBoost,0.9912,0.9474,0.9906,0.9429,0.1501,117.0,1.4112,0.0985,158.3232
3,LightGBM,0.9934,0.9386,0.9930,0.9337,0.0337,117.0,0.8670,0.0936,132.2725
4,CatBoost,0.9934,0.9561,0.9930,0.9521,0.3430,155.0,0.9501,0.0598,67.7031


In [16]:
print("=== CatBoost на категориальных признаках ===")

X_cat = X.copy()
for col in X_cat.columns:
    X_cat[col] = pd.cut(X_cat[col], bins=10, labels=False).astype(int)

X_train_cat, X_test_cat, y_train_cat, y_test_cat = train_test_split(
    X_cat, y, test_size=0.2, random_state=67, stratify=y
)

cat_model_cat = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=4,
    random_seed=67,
    verbose=False,
    early_stopping_rounds=50,
    thread_count=-1,
    cat_features=list(range(X_cat.shape[1]))
)

X_train_val_cat, X_val_cat, y_train_val_cat, y_val_cat = train_test_split(
    X_train_cat, y_train_cat, test_size=0.2, random_state=67, stratify=y_train_cat
)

start = time.time()
cat_model_cat.fit(
    X_train_val_cat, y_train_val_cat,
    eval_set=(X_val_cat, y_val_cat),
    verbose=False
)
cat_cat_time = time.time() - start

cat_cat_pred_test = cat_model_cat.predict(X_test_cat)
cat_cat_accuracy = accuracy_score(y_test_cat, cat_cat_pred_test)
cat_cat_f1 = f1_score(y_test_cat, cat_cat_pred_test, average="macro")

print(f"CatBoost с категориальными признаками (native):")
print(f"  Accuracy test: {cat_cat_accuracy:.4f}")
print(f"  F1 macro test: {cat_cat_f1:.4f}")
print(f"  Время обучения: {cat_cat_time:.3f} сек")
print(f"  Лучшая итерация: {cat_model_cat.best_iteration_}")

=== CatBoost на категориальных признаках ===


CatBoost с категориальными признаками (native):
  Accuracy test: 0.9123
  F1 macro test: 0.9048
  Время обучения: 2.862 сек
  Лучшая итерация: 54


In [17]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_train_cat_ohe = encoder.fit_transform(X_train_cat)
X_val_cat_ohe = encoder.transform(X_val_cat)
X_test_cat_ohe = encoder.transform(X_test_cat)

xgb_model_cat = xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=4,
    random_state=67,
    n_jobs=-1,
    eval_metric="logloss",
    early_stopping_rounds=50,
    verbosity=0
)

start = time.time()
xgb_model_cat.fit(
    X_train_cat_ohe, y_train_cat,
    eval_set=[(X_val_cat_ohe, y_val_cat)],
    verbose=False
)
xgb_cat_time = time.time() - start

xgb_cat_pred_test = xgb_model_cat.predict(X_test_cat_ohe)
xgb_cat_accuracy = accuracy_score(y_test_cat, xgb_cat_pred_test)
xgb_cat_f1 = f1_score(y_test_cat, xgb_cat_pred_test, average="macro")

print(f"XGBoost с one-hot кодированием:")
print(f"  Accuracy test: {xgb_cat_accuracy:.4f}")
print(f"  F1 macro test: {xgb_cat_f1:.4f}")
print(f"  Время обучения: {xgb_cat_time:.3f} сек")
print(f"  Лучшая итерация: {xgb_model_cat.best_iteration}")
print(f"  Признаков после OHE: {X_train_cat_ohe.shape[1]}")

XGBoost с one-hot кодированием:
  Accuracy test: 0.9474
  F1 macro test: 0.9429
  Время обучения: 1.083 сек
  Лучшая итерация: 998
  Признаков после OHE: 272


In [18]:
lgb_model_cat = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=4,
    random_state=67,
    n_jobs=-1,
    verbosity=-1,
    force_col_wise=True
)

start = time.time()
lgb_model_cat.fit(
    X_train_cat_ohe, y_train_cat,
    eval_set=[(X_val_cat_ohe, y_val_cat)],
    eval_metric="logloss",
    callbacks=[lgb.early_stopping(50, verbose=False)]
)
lgb_cat_time = time.time() - start

lgb_cat_pred_test = lgb_model_cat.predict(X_test_cat_ohe)
lgb_cat_accuracy = accuracy_score(y_test_cat, lgb_cat_pred_test)
lgb_cat_f1 = f1_score(y_test_cat, lgb_cat_pred_test, average="macro")

print(f"LightGBM с one-hot кодированием:")
print(f"  Accuracy test: {lgb_cat_accuracy:.4f}")
print(f"  F1 macro test: {lgb_cat_f1:.4f}")
print(f"  Время обучения: {lgb_cat_time:.3f} сек")
print(f"  Лучшая итерация: {lgb_model_cat.best_iteration_}")
print(f"  Признаков после OHE: {X_train_cat_ohe.shape[1]}")

C:\Users\user\Desktop\projects\MLProjectsCS32\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


LightGBM с one-hot кодированием:
  Accuracy test: 0.9561
  F1 macro test: 0.9526
  Время обучения: 1.183 сек
  Лучшая итерация: 1000
  Признаков после OHE: 272


In [19]:
cat_original_results = {
    "model": "CatBoost (исх. числовые)",
    "accuracy_test": cat_results["accuracy_test"],
    "f1_macro_test": cat_results["f1_macro_test"],
    "train_time": cat_results["train_time"]
}

cat_categorical_results = {
    "model": "CatBoost (категориальные, native)",
    "accuracy_test": cat_cat_accuracy,
    "f1_macro_test": cat_cat_f1,
    "train_time": cat_cat_time
}

xgb_original_results = {
    "model": "XGBoost (исх. числовые)",
    "accuracy_test": xgb_results["accuracy_test"],
    "f1_macro_test": xgb_results["f1_macro_test"],
    "train_time": xgb_results["train_time"]
}

xgb_categorical_results = {
    "model": "XGBoost (one-hot)",
    "accuracy_test": xgb_cat_accuracy,
    "f1_macro_test": xgb_cat_f1,
    "train_time": xgb_cat_time
}

lgb_original_results = {
    "model": "LightGBM (исх. числовые)",
    "accuracy_test": lgb_results["accuracy_test"],
    "f1_macro_test": lgb_results["f1_macro_test"],
    "train_time": lgb_results["train_time"]
}

lgb_categorical_results = {
    "model": "LightGBM (one-hot)",
    "accuracy_test": lgb_cat_accuracy,
    "f1_macro_test": lgb_cat_f1,
    "train_time": lgb_cat_time
}

categorical_comparison = pd.DataFrame([
    cat_original_results, cat_categorical_results,
    xgb_original_results, xgb_categorical_results,
    lgb_original_results, lgb_categorical_results
])

display(categorical_comparison.round(4))

,model,accuracy_test,f1_macro_test,train_time
0,CatBoost (исх. числовые),0.9561,0.9521,0.3430
1,"CatBoost (категориальные, native)",0.9123,0.9048,2.8617
2,XGBoost (исх. числовые),0.9474,0.9429,0.1501
3,XGBoost (one-hot),0.9474,0.9429,1.0831
4,LightGBM (исх. числовые),0.9386,0.9337,0.0337
5,LightGBM (one-hot),0.9561,0.9526,1.1832


In [20]:
summary = {
    "dataset": "Breast Cancer Wisconsin",
    "n_samples": int(X.shape[0]),
    "n_features": int(X.shape[1]),
    "test_size": 0.2,
    "random_state": 67,
    "val_size_from_train": 0.2,
    "libraries": {
        "xgboost": xgb.__version__,
        "lightgbm": lgb.__version__,
        "catboost": __import__('catboost').__version__,
        "scikit-learn": __import__('sklearn').__version__
    },
    "results": full_comparison.to_dict(orient="records"),
    "categorical_results": categorical_comparison.to_dict(orient="records")
}

with open("../results_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Результаты сохранены в ../results_summary.json")

Результаты сохранены в ../results_summary.json
